# Named-entity recognition and clinical coding: worked solution

Worked solution for the exercise in the lesson **[Named-entity recognition and clinical coding](https://ramkumarmd.com/medical-ai/clinical-nlp/clinical-ner-and-coding)** (*Clinical NLP* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](02-clinical-ner-and-coding.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Named-entity recognition (NER) finds mentions of things such as conditions, medicines and procedures in text; normalisation maps each mention to a standard code, such as ICD-10 or SNOMED CT. Dictionary methods are transparent and fast but miss synonyms and spelling variants; machine-learning and transformer models find more but need labelled data and validation. Any extractor must handle negation and be scored against human annotations with precision and recall. Studies have found that large language models are unreliable at assigning medical codes directly.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q medspacy

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### A dictionary extractor with negation

Dictionary-based extraction to ICD-10 categories, scored with and without negation handling.

In [ ]:
import re

# Illustrative WHO ICD-10 categories; always confirm codes in the official classification
LEXICON = {
    "E11": ["type 2 diabetes", "t2dm", "type ii diabetes"],
    "I10": ["hypertension", "htn", "high blood pressure"],
    "N18": ["chronic kidney disease", "ckd"],
    "L97": ["foot ulcer", "ulcer of the foot", "ulcers"],
    "E78": ["hyperlipidaemia", "dyslipidaemia", "high cholesterol"],
}
NEGATION = r"\b(no|denies|without|negative for|absent)\b"

def extract_codes(note, use_negation=True):
    codes = set()
    for sentence in re.split(r"(?<=[.!?])\s+", note.lower()):
        for code, terms in LEXICON.items():
            for term in terms:
                m = re.search(rf"\b{re.escape(term)}\b", sentence)
                if m and not (use_negation and re.search(NEGATION, sentence[:m.start()])):
                    codes.add(code)
    return codes

NOTES = [   # synthetic notes with the codes a (fictional) clinical coder assigned
    ("58M with T2DM and HTN. No foot ulcers. Neuropathy on monofilament testing.", {"E11", "I10"}),
    ("Type 2 diabetes, CKD stage 3. Denies chest pain. High cholesterol on statin.", {"E11", "N18", "E78"}),
    ("Known DM2 for 6 years. Left heel ulcer, 2 cm, no signs of infection.", {"E11", "L97"}),
    ("Hypertension well controlled. Negative for diabetes on OGTT.", {"I10"}),
]

def score(use_negation):
    tp = fp = fn = 0
    for text, gold in NOTES:
        pred = extract_codes(text, use_negation)
        tp, fp, fn = tp + len(pred & gold), fp + len(pred - gold), fn + len(gold - pred)
    return tp / (tp + fp), tp / (tp + fn)

for label, neg in [("Without negation handling", False), ("With negation handling", True)]:
    precision, recall = score(neg)
    print(f"{label:<27} precision {precision:.2f}  recall {recall:.2f}")

# Without negation handling   precision 0.86  recall 0.75
# With negation handling      precision 1.00  recall 0.75

### Where dictionaries fail

Continuing from the previous block: list the notes where extraction disagrees with the coder.

In [ ]:
for text, gold in NOTES:
    pred = extract_codes(text)
    if pred != gold:
        print(f"{text}\n  missed {sorted(gold - pred)}  extra {sorted(pred - gold)}")

# Known DM2 for 6 years. Left heel ulcer, 2 cm, no signs of infection.
#   missed ['E11', 'L97']  extra []

## Exercise data

Synthetic data written for this exercise (no real patients). Run this cell before starting the steps.

In [ ]:
# Thirty SYNTHETIC notes with the codes a (fictional) clinical coder assigned, using the five illustrative
# ICD-10 categories of the lesson's lexicon (E11, I10, N18, L97, E78). No real patients.
CODED_NOTES = [
    ("58M with T2DM and HTN. No foot ulcers.", {"E11", "I10"}),
    ("Type 2 diabetes, CKD stage 3. High cholesterol on statin.", {"E11", "N18", "E78"}),
    ("Known DM2 for 6 years. Left heel ulcer, 2 cm.", {"E11", "L97"}),
    ("Hypertension well controlled. Negative for diabetes on OGTT.", {"I10"}),
    ("Type II diabetes mellitus. eGFR 48, chronic kidney disease.", {"E11", "N18"}),
    ("Raised BP readings, diagnosed hypertension today.", {"I10"}),
    ("No history of diabetes. Dyslipidaemia, started atorvastatin.", {"E78"}),
    ("Foot ulcer, right hallux, healing. T2DM.", {"E11", "L97"}),
    ("T2DM on metformin. Denies high blood pressure. Lipids normal.", {"E11"}),
    ("Mother has type 2 diabetes. Patient's HbA1c 5.4%, normal.", set()),
    ("Possible CKD, repeat eGFR in 3 months.", set()),
    ("Type 2 DM with nephropathy, CKD 4.", {"E11", "N18"}),
    ("Essential hypertension. Hyperlipidaemia.", {"I10", "E78"}),
    ("Ulceration of the left foot over the 5th metatarsal head. T2DM.", {"E11", "L97"}),
    ("Type 2 diabetes. HTN. CKD 3a. High cholesterol.", {"E11", "I10", "N18", "E78"}),
    ("Pre-diabetes on HbA1c. No hypertension.", set()),
    ("Non-insulin-dependent diabetes since 2015.", {"E11"}),
    ("Raised cholesterol, family history of heart disease.", {"E78"}),
    ("Kidney function reduced, stage 3 chronic kidney disease. No diabetes.", {"N18"}),
    ("Neuropathic ulcer on the sole of the right foot. Type 2 diabetes.", {"E11", "L97"}),
    ("Father had hypertension. Patient's BP 118/76.", set()),
    ("T2DM, well controlled. Foot check: no ulcers, pulses present.", {"E11"}),
    ("HTN on amlodipine. Query CKD, awaiting urine ACR.", {"I10"}),
    ("Type 2 diabetes with foot ulcer and high blood pressure.", {"E11", "L97", "I10"}),
    ("Hypercholesterolaemia. Statin declined.", {"E78"}),
    ("Diabetes type 2, insulin-treated. Chronic renal failure stage 4.", {"E11", "N18"}),
    ("Rule out hypertension: home BP monitoring arranged.", set()),
    ("DM2. HTN. Dyslipidemia.", {"E11", "I10", "E78"}),
    ("Ulcer on the heel, infected. Type 2 diabetes.", {"E11", "L97"}),
    ("Patient without diabetes. CKD stage 2 from polycystic kidneys.", {"N18"}),
]
print(f"{len(CODED_NOTES)} coded synthetic notes")

## Exercise

**Goal:** Build, score and improve a clinical entity extractor on synthetic notes.

**Data:** Thirty synthetic notes with coder-style labels, written for the notebook (Created for this course; no real patient data)

### Step 1

Reproduce the extractor and scores, then run it on the thirty notes and report precision and recall per code.

In [ ]:
import pandas as pd

CODES = list(LEXICON)

def per_code_scores(extractor, notes=CODED_NOTES):
    rows = []
    for code in CODES:
        tp = sum(code in extractor(t) and code in g for t, g in notes)
        fp = sum(code in extractor(t) and code not in g for t, g in notes)
        fn = sum(code not in extractor(t) and code in g for t, g in notes)
        rows.append({"code": code, "TP": tp, "FP": fp, "FN": fn,
                     "precision": tp / (tp + fp) if tp + fp else float("nan"), "recall": tp / (tp + fn) if tp + fn else float("nan")})
    df = pd.DataFrame(rows).set_index("code")
    tp, fp, fn = df.TP.sum(), df.FP.sum(), df.FN.sum()
    print(df.round(2).to_string())
    print(f"overall precision {tp / (tp + fp):.2f}, recall {tp / (tp + fn):.2f}")
    return df

base = per_code_scores(extract_codes)
print("\nErrors:")
for text, gold in CODED_NOTES:
    pred = extract_codes(text)
    if pred != gold:
        print(f"  missed {sorted(gold - pred)} extra {sorted(pred - gold)}  | {text}")

### Step 2

Add synonyms and abbreviations from your error list and re-score; did precision fall?

In [ ]:
import copy

LEXICON_V1 = copy.deepcopy(LEXICON)
LEXICON["E11"] += ["dm2", "type 2 dm", "diabetes type 2", "type ii diabetes mellitus", "non-insulin-dependent diabetes"]
LEXICON["I10"] += ["essential hypertension", "raised bp"]
LEXICON["N18"] += ["chronic renal failure", "ckd 3", "ckd 3a", "ckd 4", "stage 3 chronic kidney disease"]
LEXICON["L97"] += ["heel ulcer", "ulcer on the heel", "ulceration of the left foot", "neuropathic ulcer",
                   "foot ulcer", "ulcer", "ulceration"]
LEXICON["E78"] += ["hypercholesterolaemia", "raised cholesterol", "hyperlipidemia", "dyslipidemia"]

after_synonyms = per_code_scores(extract_codes)

Synonyms take recall from 0.70 to 1.00 here, and precision does not fall (0.86 to 0.90): the remaining false
positives are the uncertain and family-history notes, not the new terms. On real notes, broad terms such as "ulcer"
(an ulcer elsewhere) or "raised BP" (one high reading, not a diagnosis) would add false positives, so every lexicon
change is re-scored on the whole labelled set.

### Step 3

Add uncertainty and family-history handling from the previous lesson and measure the effect.

In [ ]:
UNCERTAIN = r"\b(possible|probable|query|\?|suspected|rule out|awaiting)\b"
FAMILY = r"\b(mother|father|sister|brother|family history)\b"

def extract_codes_v2(note):
    codes = set()
    for sentence in re.split(r"(?<=[.!?])\s+", note.lower()):
        for code, terms in LEXICON.items():
            for term in terms:
                m = re.search(rf"\b{re.escape(term)}\b", sentence)
                if not m:
                    continue
                before = sentence[:m.start()]
                if re.search(NEGATION, before) or re.search(UNCERTAIN, before) or re.search(FAMILY, sentence):
                    continue
                codes.add(code)
    return codes

after_context = per_code_scores(extract_codes_v2)
for text, gold in CODED_NOTES:
    pred = extract_codes_v2(text)
    if pred != gold:
        print(f"  still wrong: missed {sorted(gold - pred)} extra {sorted(pred - gold)}  | {text}")

Uncertainty and family-history handling remove every false positive ("possible CKD", "query CKD", "rule out
hypertension", "mother has type 2 diabetes", "father had hypertension"): precision 1.00, recall 0.98. The one new
miss shows the cost of a blunt rule: "Raised cholesterol, family history of heart disease" loses the patient's own
cholesterol because the family cue silences the whole sentence. A better rule limits the cue's scope to the words
that follow it, as ConText does.

### Step 4

Optional: run a scispaCy or medspaCy model on the same notes and compare.

In [ ]:
# Optional: medspaCy (pip install medspacy) with our lexicon as target rules and its ConText module for negation,
# uncertainty and family history. scispaCy's biomedical NER models are an alternative; they are installed from
# version-matched model files listed in the scispaCy documentation.
import medspacy
from loguru import logger
from medspacy.target_matcher import TargetRule

logger.remove()
nlp = medspacy.load()
def rule(term):
    # Single-word terms as token patterns: in our run, a plain literal rule for "t2dm" did not match "T2DM"
    words = term.split()
    return TargetRule(term, "CONDITION", pattern=[{"LOWER": w} for w in words]) if len(words) == 1 else TargetRule(term, "CONDITION")
nlp.get_pipe("medspacy_target_matcher").add([rule(term) for terms in LEXICON.values() for term in terms])

def extract_codes_medspacy(note):
    doc = nlp(note)
    return {next(code for code, terms in LEXICON.items() if e.text.lower() in terms)
            for e in doc.ents if not (e._.is_negated or e._.is_uncertain or e._.is_family or e._.is_hypothetical)}

after_medspacy = per_code_scores(extract_codes_medspacy)
for text, gold in CODED_NOTES:
    pred = extract_codes_medspacy(text)
    if pred != gold:
        doc = nlp(text)
        print(f"  medspaCy: missed {sorted(gold - pred)} extra {sorted(pred - gold)} | {text} | entities "
              f"{[(e.text, 'neg' if e._.is_negated else 'unc' if e._.is_uncertain else 'fam' if e._.is_family else 'hyp' if e._.is_hypothetical else 'ok') for e in doc.ents]}")

Running a library is not the same as validating it. In our first run, medspaCy missed every "T2DM" (a plain literal
rule did not match it; the token-pattern rule above fixes that). Read the printed disagreements: in our run ConText
also did not treat "Query CKD" as uncertain, and did not negate "no ulcers" in "Foot check: no ulcers, pulses
present.", although it did negate "No ulcers." on its own. Its curated cue lists help most on long, messy notes
(cue scopes, post-position cues, terminating terms), but every local abbreviation and cue has to be added and
tested. Neither approach knows which terms count as a codable diagnosis; that sits in the lexicon and coding rules.

### Step 5

Optional, with an approved model: ask an LLM to extract conditions (not codes), map them to codes with your lexicon, and compare with direct LLM coding.

Optional, with an approved model and these synthetic notes only. Two designs to compare:

```python
def ask(prompt: str) -> str:
    """Call your approved model; return its text (API key from an environment variable)."""
    raise NotImplementedError

EXTRACT = ("List the conditions this synthetic note says the patient currently HAS (not negated, not uncertain, "
           "not family history), one per line, in plain words. Note: {note}")
CODE = ("Give the ICD-10 category codes (three characters) for conditions the patient currently has, "
        "comma-separated, from this list only: E11, I10, N18, L97, E78. Note: {note}")

def via_lexicon(note):                       # LLM extracts conditions, our lexicon maps them to codes
    return set().union(*(extract_codes_v2(line) for line in ask(EXTRACT.format(note=note)).splitlines()))

def direct(note):                            # LLM assigns codes itself
    return {c.strip() for c in ask(CODE.format(note=note)).split(",") if c.strip() in LEXICON}

per_code_scores(via_lexicon); per_code_scores(direct)
```

Expect the LLM to handle negation and paraphrase well, and to make different errors: occasionally coding
conditions that are implied but not stated, or inventing codes outside the list (which the filter removes). Mapping
through a reviewed lexicon keeps the final codes auditable.

---

Lesson: [https://ramkumarmd.com/medical-ai/clinical-nlp/clinical-ner-and-coding](https://ramkumarmd.com/medical-ai/clinical-nlp/clinical-ner-and-coding). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.